# RES CF: area-weighting, the P95 site-selection question, and coastal sea-contamination

Exploratory analysis behind the PR #8 review (area-weighted P95 fix in script 07).

**Question that started it:** how much does area-weighting change the selected
"P95 best-site" capacity factor, and is that difference real signal or an artifact?

**Conclusions (established below):**
1. The weighted-vs-unweighted P95 difference is **negligible for solar & offshore**
   but **first-order for onshore wind** (≈ a full P90→P95 percentile step).
2. atlite's `indicatormatrix` weight is a **land-coverage fraction** (EPSG:4326,
   0.25 deg² per cell) — **no cos(lat)**, so it is not a true physical-area weight.
3. The onshore divergence is **~100% a sea/land border effect**: coastal cells that
   straddle the coast are windier and dominate the high-CF tail; drop them and the
   two methods agree.
4. Decomposition shows the coastal overperformance is **mostly the sea *in the cell*
   (a coarse-ERA5 artifact), not coastal *land* being windier** — often coastal land
   is *worse* than inland.
5. This is a textbook reanalysis artifact; the real fix is a land-sea mask
   (PyPSA-Eur style) or a higher-resolution wind product, not the weighting knob.

> Run top-to-bottom. The **Heavy compute pass** cell builds CF grids for all
> cutouts once (~10–15 min); every later cell reuses the cache.

## Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import atlite
import yaml
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "notebook"  # embed interactive figures in the notebook

# Resolve repo root whether run from repo root or from notebooks/
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

CFG = yaml.safe_load(open(ROOT / "config/config.yaml"))["res_cf"]
WCF = CFG.get("wind_cf", {})

# (cf_area, cutout file, region code). All 2023 except the committed VIC demo (2025).
COUNTRIES = [
    ("fr", "fr_20230101_20231231.nc", "FR"),
    ("de", "de_20230101_20231231.nc", "DE"),
    ("es", "es_20230101_20231231.nc", "ES"),
    ("aus", "aus_20230101_20231231.nc", "AUS"),
    ("bra", "bra_20230101_20231231.nc", "BRA"),
    ("vic", "vic_20250101_20251231.nc", "VIC"),
]
TECHS = ["wind_onshore", "wind_offshore", "solar"]
TECH_COLOR = {"wind_onshore": "#2E7D32", "wind_offshore": "#1565C0", "solar": "#E8833A"}
TECH_SHORT = {"wind_onshore": "onshore", "wind_offshore": "offshore", "solar": "solar"}

In [ ]:
def weighted_percentile(values, weights, q):
    # Area-weighted percentile (q in [0,1]); returns an actual data value.
    m = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    v, w = values[m], weights[m]
    order = np.argsort(v); v, w = v[order], w[order]
    cw = np.cumsum(w); cw /= cw[-1]
    return float(v[min(np.searchsorted(cw, q, side="left"), v.size - 1)])


def haversine(lon1, lat1, lon2, lat2):
    R = 6371.0
    lon1, lat1, lon2, lat2 = map(np.deg2rad, (lon1, lat1, lon2, lat2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return R * 2 * np.arcsin(np.sqrt(a))


def cf_hourly(co, tech):
    # Per-cell hourly CF grid (time, y, x) matching script 07's technology setup.
    if tech == "solar":
        cf = co.pv(panel=CFG["pv_panel"], orientation=CFG["pv_orientation"],
                   capacity_factor_timeseries=True)
    else:
        turb = CFG["wind_onshore_turbine"] if tech == "wind_onshore" else CFG["wind_offshore_turbine"]
        cf = co.wind(turbine=turb, capacity_factor_timeseries=True,
                     smooth=WCF.get("smooth", True),
                     add_cutout_windspeed=WCF.get("add_cutout_windspeed", True))
    if hasattr(cf, "data_vars"):
        cf = cf[list(cf.data_vars)[0]]
    return cf

## Heavy compute pass (run once)

For every (country, tech) we cache the flat per-cell arrays used by all analyses:
annual-mean CF, std of daily-mean CF, indicatormatrix land-coverage weight, and
cell lon/lat. Onshore/solar use the land geometry; offshore uses the EEZ geometry.

In [ ]:
CACHE = {}  # (cc, tech) -> dict(mean, std_daily, w, lon, lat, region)
for cc, fname, region in COUNTRIES:
    cpath = ROOT / "cutouts" / fname
    if not cpath.exists():
        print(f"SKIP {cc}: missing {fname}"); continue
    co = atlite.Cutout(path=str(cpath))
    land = gpd.read_parquet(ROOT / f"resources/shapes/{cc}_geo.parquet")
    off = gpd.read_parquet(ROOT / f"resources/shapes/{cc}_offshore_geo.parquet")
    xs, ys = co.data.x.values, co.data.y.values
    xx, yy = np.meshgrid(xs, ys)
    lon, lat = xx.ravel(), yy.ravel()
    for tech in TECHS:
        gdf = off if tech == "wind_offshore" else land
        geom = gdf.loc[gdf["region"] == region].geometry.iloc[0]
        cfh = cf_hourly(co, tech)
        mean = cfh.mean("time").values.ravel()
        std_daily = cfh.resample(time="1D").mean("time").std("time").values.ravel()
        ind = co.indicatormatrix([geom]).tocsr()
        w = np.asarray(ind[0, :].todense()).ravel()
        CACHE[(cc, tech)] = dict(mean=mean, std_daily=std_daily, w=w,
                                 lon=lon, lat=lat, region=region)
        print(f"{region} {tech}: {(w > 0).sum()} in-region cells")
print("done")

## 1. Weighted vs unweighted P95 (annual-mean CF)

Same cell set (indicatormatrix weight > 0); weighted percentile (as in 07's
`find_p95_cell`) vs plain unweighted. `d_p95` = the weighting effect on the value;
`sel_*` = annual-mean CF of the cell each method actually selects (what flows
downstream); `diff_cell` = whether a different cell is picked.

In [ ]:
rows = []
for (cc, tech), d in CACHE.items():
    v, w = d["mean"], d["w"]
    valid = (w > 0) & np.isfinite(v) & (v > 0)
    vv, ww = v[valid], w[valid]
    p95u = float(np.percentile(vv, 95))
    p95w = weighted_percentile(vv, ww, 0.95)
    sel_u = float(vv[np.argmin(np.abs(vv - p95u))])
    sel_w = float(vv[np.argmin(np.abs(vv - p95w))])
    rows.append(dict(country=d["region"], tech=tech, n_cells=int(valid.sum()),
                     p95_unw=p95u, p95_wtd=p95w, d_p95=p95u - p95w,
                     sel_unw=sel_u, sel_wtd=sel_w))
p95_tbl = pd.DataFrame(rows)
p95_tbl.round(4)

Solar & offshore differences are ~0; the effect is an **onshore-wind** phenomenon,
and it always *lowers* the P95 (weighted < unweighted).

## 2. Magnitude in perspective: weighting effect vs the P90→P95 step

Is the weighting effect big? Compare it to a natural yardstick — how much CF moves
for one percentile step (P90→P95). `ratio = d_weight / d_pctile` expresses the
weighting effect in "percentile-steps".

In [ ]:
rows = []
for (cc, tech), d in CACHE.items():
    v, w = d["mean"], d["w"]
    valid = (w > 0) & np.isfinite(v) & (v > 0)
    vv, ww = v[valid], w[valid]
    p90u = float(np.percentile(vv, 90)); p95u = float(np.percentile(vv, 95))
    p95w = weighted_percentile(vv, ww, 0.95)
    d_pctile = p95u - p90u; d_weight = p95u - p95w
    rows.append(dict(country=d["region"], tech=tech, P90=p90u, P95_unw=p95u,
                     P95_wtd=p95w, d_pctile=d_pctile, d_weight=d_weight,
                     ratio=(d_weight / d_pctile if d_pctile else np.nan)))
persp = pd.DataFrame(rows)
persp[persp.tech == "wind_onshore"].round(4)

In [ ]:
# Solar & offshore for completeness — both effects negligible
persp[persp.tech != "wind_onshore"].round(4)

For onshore wind the weighting effect is a large fraction of a full P90→P95 step
(ratio ~0.35–1.2, median ~0.7) — a first-order choice. For solar/offshore both are
tiny.

## 3. Percentile curve with ±2σ daily band — FR onshore wind

Line = annual-mean-CF quantile function (per weighting method). Error band at each
percentile = ±2σ of the *representative cell's daily-mean CF* over the year
(intra-annual variability). The two methods track together until the upper tail,
where P95 lives.

In [ ]:
def curve(vv, ww, ss, pcts, weighted):
    ys, err = [], []
    for p in pcts:
        target = (weighted_percentile(vv, ww, p / 100.0) if weighted
                  else float(np.percentile(vv, p)))
        i = int(np.argmin(np.abs(vv - target)))
        ys.append(float(vv[i])); err.append(2.0 * float(ss[i]))
    return np.array(ys), np.array(err)


d = CACHE[("fr", "wind_onshore")]
valid = (d["w"] > 0) & np.isfinite(d["mean"]) & (d["mean"] > 0)
vv, ww, ss = d["mean"][valid], d["w"][valid], d["std_daily"][valid]
pcts = np.arange(1, 100)

fig = go.Figure()
for weighted, name, color, rgba in [
    (False, "unweighted (per-cell)", "#2E5EAA", "rgba(46,94,170,0.15)"),
    (True, "area-weighted", "#E8833A", "rgba(232,131,58,0.15)"),
]:
    y, e = curve(vv, ww, ss, pcts, weighted)
    lo = np.clip(y - e, 0, 1); hi = np.clip(y + e, 0, 1)
    fig.add_trace(go.Scatter(x=np.r_[pcts, pcts[::-1]], y=np.r_[hi, lo[::-1]],
                             fill="toself", fillcolor=rgba, line=dict(width=0),
                             hoverinfo="skip", showlegend=False))
    fig.add_trace(go.Scatter(x=pcts, y=y, name=name, line=dict(color=color, width=2.5)))
fig.add_vline(x=95, line=dict(color="grey", dash="dot"), annotation_text="P95")
fig.update_layout(title="FR onshore wind — annual-mean CF vs percentile (band = ±2σ daily-mean CF)",
                  xaxis_title="percentile", yaxis_title="annual-mean capacity factor",
                  template="plotly_white", width=950, height=580)
fig.update_yaxes(range=[0, 1])
fig

## 4. All countries & techs — click through by country

Legend grouped by country; color = tech, solid = unweighted / dotted = area-weighted.
±2σ daily band as symmetric error bars (so it toggles with the line). Only FR
onshore shown by default — click legend entries to add others.

In [ ]:
PCTS = np.arange(3, 100, 4)
fig = go.Figure()
for cc, fname, region in COUNTRIES:
    for tech in TECHS:
        if (cc, tech) not in CACHE:
            continue
        d = CACHE[(cc, tech)]
        valid = (d["w"] > 0) & np.isfinite(d["mean"]) & (d["mean"] > 0)
        vv, ww, ss = d["mean"][valid], d["w"][valid], d["std_daily"][valid]
        color = TECH_COLOR[tech]
        for weighted, method, dash in [(False, "unweighted", "solid"),
                                       (True, "area-weighted", "dot")]:
            y, e = curve(vv, ww, ss, PCTS, weighted)
            vis = True if (cc == "fr" and tech == "wind_onshore") else "legendonly"
            fig.add_trace(go.Scatter(
                x=PCTS, y=y, name=f"{TECH_SHORT[tech]} · {method}",
                legendgroup=region, legendgrouptitle_text=region,
                mode="lines+markers", visible=vis,
                line=dict(color=color, width=2, dash=dash),
                marker=dict(size=3, color=color),
                error_y=dict(type="data", array=e, thickness=0.8, width=0, color=color)))
fig.update_layout(title="Annual-mean CF vs percentile by country & tech (±2σ daily band)",
                  xaxis_title="percentile", yaxis_title="annual-mean capacity factor",
                  template="plotly_white", width=1050, height=720,
                  legend=dict(groupclick="toggleitem", tracegroupgap=12))
fig.update_yaxes(range=[0, 1])
fig

## 5. Is the divergence a sea/land border effect? (onshore wind)

`indicatormatrix` is a land-coverage fraction in [0,1] (`w_max`≈1, cells 0.25 deg²,
**no cos(lat)**). Classify cells: full-land (w≥0.98) vs border/straddle (0<w<0.98).
- `corr(w,CF)` < 0 → less-land (more-sea) cells are windier.
- `CFratio_border` > 1 → border cells beat their nearest full-land neighbour.
- `tail%border` → share of the top-5% CF cells that straddle.
- `gap_full` vs `gap_noborder` / `explained%` → recompute the P95 gap on full-land
  cells only; if it collapses, the divergence is entirely coastal.

In [ ]:
rows = []
for (cc, tech), d in CACHE.items():
    if tech != "wind_onshore":
        continue
    v, w, lon, lat = d["mean"], d["w"], d["lon"], d["lat"]
    inreg = (w > 0) & np.isfinite(v) & (v > 0)
    wmax = float(w[inreg].max())
    full = inreg & (w >= 0.98 * wmax)
    border = inreg & (w < 0.98 * wmax)
    corr = float(np.corrcoef(w[inreg], v[inreg])[0, 1])
    fl = np.where(full)[0]
    ratios = []
    for i in np.where(border)[0]:
        j = fl[int(np.argmin(haversine(lon[i], lat[i], lon[fl], lat[fl])))]
        if v[j] > 0:
            ratios.append(v[i] / v[j])
    p95u = float(np.percentile(v[inreg], 95))
    tail = inreg & (v >= p95u)
    gap_full = p95u - weighted_percentile(v[inreg], w[inreg], 0.95)
    gap_nb = float(np.percentile(v[full], 95)) - weighted_percentile(v[full], w[full], 0.95)
    rows.append(dict(country=d["region"], n_in=int(inreg.sum()),
                     pct_border=round(100 * border.sum() / inreg.sum(), 1),
                     corr_w_CF=round(corr, 3),
                     CFratio_border=round(float(np.median(ratios)), 3),
                     tail_pct_border=round(100 * (tail & border).sum() / max(1, tail.sum()), 1),
                     gap_full=round(gap_full, 4), gap_noborder=round(gap_nb, 4),
                     explained_pct=round(100 * (1 - gap_nb / gap_full), 1) if gap_full else np.nan))
pd.DataFrame(rows)

`explained% ≈ 100` in every country: removing border cells collapses the P95 gap to
≈0. The weighted-vs-unweighted divergence is **entirely** a coastal border effect.

## 6. Decompose the coastal overperformance: sea-in-cell vs coastal-land

Three groups (onshore): deep-inland full-land (baseline), coast-adjacent full-land
(pure land, coastal), border cells (contain sea). Overperformance vs inland splits
exactly: `(border − inland) = (coast_adj − inland) + (border − coast_adj)`, i.e.
**coastal-land effect** + **sea-in-cell effect**.

In [ ]:
D_ADJ, D_INLAND = 75.0, 200.0  # km
rows = []
for (cc, tech), d in CACHE.items():
    if tech != "wind_onshore":
        continue
    v, w, lon, lat = d["mean"], d["w"], d["lon"], d["lat"]
    inreg = (w > 0) & np.isfinite(v) & (v > 0)
    full = inreg & (w >= 0.98); border = inreg & (w < 0.98)
    b_lon, b_lat = lon[border], lat[border]
    f_idx = np.where(full)[0]
    dmin = np.array([haversine(lon[i], lat[i], b_lon, b_lat).min() if b_lon.size else 1e9
                     for i in f_idx])
    coast_ad = f_idx[dmin <= D_ADJ]; inland = f_idx[dmin > D_INLAND]
    cf_in = float(np.mean(v[inland])) if inland.size else np.nan
    cf_ca = float(np.mean(v[coast_ad])) if coast_ad.size else np.nan
    cf_bd = float(np.mean(v[border])) if border.any() else np.nan
    coastal_land = cf_ca - cf_in; sea_cell = cf_bd - cf_ca; total = cf_bd - cf_in
    slope = float(np.polyfit(1.0 - w[border], v[border], 1)[0]) if border.sum() > 2 else np.nan
    rows.append(dict(country=d["region"], CF_inland=round(cf_in, 3),
                     CF_coastLand=round(cf_ca, 3), CF_border=round(cf_bd, 3),
                     coastal_land=round(coastal_land, 4), sea_in_cell=round(sea_cell, 4),
                     total=round(total, 4),
                     land_share_pct=round(100 * coastal_land / total, 1) if total else np.nan,
                     sea_share_pct=round(100 * sea_cell / total, 1) if total else np.nan,
                     seaSlope=round(slope, 3)))
pd.DataFrame(rows)

The **sea-in-cell** term dominates (63–162% of the overperformance) and `seaSlope`>0
everywhere: border-cell CF rises with sea fraction. Coastal **land** is often *worse*
than inland (FR, ES, AUS negative). So the coastal high CF is a coarse-ERA5 sea
artifact, not buildable onshore resource — a turbine on the land fraction would see
≈ the coastal-land CF, not the inflated cell value.

## 7. Side note: area averages omit the cos(lat) factor

Because `indicatormatrix` is computed in EPSG:4326 (0.25 deg² per cell, no cos-lat),
`national_mean` is a *degree-area* average, not a physical-area average. Compare
current weighting vs `+cos(lat)`:

In [ ]:
rows = []
for (cc, tech), d in CACHE.items():
    v, w, lat = d["mean"], d["w"], d["lat"]
    m = (w > 0) & np.isfinite(v) & (v > 0)
    vv, ww, cl = v[m], w[m], np.cos(np.deg2rad(lat[m]))
    mean_deg = float((ww * vv).sum() / ww.sum())
    mean_area = float((ww * cl * vv).sum() / (ww * cl).sum())
    rows.append(dict(country=d["region"], tech=tech,
                     mean_deg=round(mean_deg, 4), mean_area=round(mean_area, 4),
                     d_rel_pct=round(100 * (mean_area - mean_deg) / mean_deg, 2)))
pd.DataFrame(rows)

Error ≤ ~2.3% (AUS offshore), usually <1% — a small, systematic bias (mildly
overstates wind, understates solar). Pre-existing; one-line fix is to multiply
weights by `cos(deg2rad(lat))` or aggregate via an equal-area CRS.

## Conclusions

- Solar & offshore: weighting is irrelevant.
- Onshore wind: weighted-vs-unweighted P95 is first-order (~a P90→P95 step), and
  ~100% a coastal sea-contamination artifact.
- The weighting knob is the wrong lever. Standard fix = a **land-sea mask /
  land-eligibility weighting** (PyPSA-Eur weights CF by eligible developable area in
  an equal-area CRS; it never picks a raw "P95 cell"). A higher-resolution/downscaled
  product (ERA5-Land, CERRA, NEWA) is the only thing that corrects the coastal CF
  *value*, since CF is capped at ERA5's ~30 km resolution.